# Fase 4: Simulación Inteligente y Optimización de Neumáticos Mineros

## Proyecto
Reducción de Variabilidad en Neumáticos Mineros CAT 797F mediante Asignación Dinámica Inteligente

## Objetivo

Desarrollar una simulación operacional sintética que permita comparar:

- un escenario tradicional de asignación fija,
vs
- un escenario inteligente basado en predicción y rotación dinámica.

La simulación integrará variables:

- geomecánicas,
- operacionales,
- térmicas,
- económicas,
- y de planeamiento,

con el objetivo de determinar la viabilidad técnica y económica de una estrategia de asignación dinámica de flota frente a una asignación fija convencional.

El horizonte de simulación será de 6 meses operacionales para una flota CAT 797F de 60 camiones.

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
np.random.seed(42)


## 1. Configuración del Sistema Operacional

Se definirán los parámetros principales del entorno operacional:

- disponibilidad mecánica,
- vida útil de neumáticos,
- límites térmicos,
- condiciones geomecánicas,
- y características de operación por tajo.

Estos parámetros servirán como base para la simulación sintética.

In [90]:
# CONFIGURACIÓN GENERAL

NUM_CAMIONES = 60
HORAS_SIMULACION = 4380
LIMITE_DESGASTE_MM = 50
COSTO_SET_LLANTAS = 312000
TKPH_LIMITE = 850
TEMP_LIMITE = 115

# TASAS DE DESGASTE REALES
# (Resultados del T-Test)

TASA_NORTE = 0.00645
TASA_SUR = 0.00834


# PARÁMETROS GEOMECÁNICOS

TAJOS = {"Norte": {"MPA": 70,"Pendiente": 10,"Factor_Desgaste": 1.0},"Sur": {"MPA": 110,"Pendiente": -10,"Factor_Desgaste": 1.30}}

print("Parámetros configurados")

Parámetros configurados


## 2. Generación de Flota Sintética

Debido a la limitada cantidad de registros históricos disponibles, se construirá una flota sintética operacionalmente consistente.

Cada camión tendrá:

- horas acumuladas,
- asignación inicial,
- TKPH,
- temperatura,
- y estado de desgaste inicial.

La simulación buscará representar un entorno operacional realista para evaluación comparativa.

In [91]:
# CREACIÓN DE FLOTA

df_flota = pd.DataFrame({"Camion_ID": [f"CAT_{i+1:03}"for i in range(NUM_CAMIONES)]})

# HORAS ACUMULADAS

df_flota["Horas_Uso_Actual"] = np.random.uniform(1500,5000,NUM_CAMIONES)

# ASIGNACIÓN ACTUAL

df_flota["Asignacion_Actual"] = np.where( np.random.rand(NUM_CAMIONES) > 0.5,"Sur","Norte")

# TKPH

df_flota["TKPH"] = np.where(df_flota["Asignacion_Actual"] == "Sur",np.random.normal(820, 40, NUM_CAMIONES),np.random.normal(650, 35, NUM_CAMIONES))


# TEMPERATURA

df_flota["Temperatura"] = np.where(df_flota["Asignacion_Actual"] == "Sur",np.random.normal(110, 6, NUM_CAMIONES),np.random.normal(92, 5, NUM_CAMIONES))

df_flota.head()

,Camion_ID,Horas_Uso_Actual,Asignacion_Actual,TKPH,Temperatura
0,CAT_001,2810.890416,Norte,626.199135,97.815819
1,CAT_002,4827.500072,Norte,658.128879,92.051165
2,CAT_003,4061.978796,Sur,804.315674,108.108385
3,CAT_004,3595.304695,Norte,624.997700,94.310517
4,CAT_005,2046.065242,Norte,715.302108,92.995298


## 3. Entrenamiento del Modelo Predictivo

Se desarrollará un modelo Random Forest para estimar el desgaste acumulado de neumáticos en función de:

- horas de operación,
- severidad geomecánica,
- TKPH,
- y temperatura operacional.

El modelo permitirá estimar el estado futuro de los neumáticos bajo distintos escenarios operacionales.

In [92]:
# DATASET SINTÉTICO DE ENTRENAMIENTO

N = 2500

X_train = pd.DataFrame({"Horas_Uso": np.random.uniform(0, 15000, N),"Es_Tajo_Sur": np.random.choice([0, 1], N),"TKPH": np.random.normal(750, 80, N),"Temperatura": np.random.normal(100, 10, N)})

# FACTOR BASE DE DESGASTE

desgaste_base = ( X_train["Horas_Uso"]*np.where(X_train["Es_Tajo_Sur"] == 1,TASA_SUR, TASA_NORTE))

# FACTOR TÉRMICO

factor_termico = np.where(X_train["TKPH"] > TKPH_LIMITE,1.20,1.0)

# FACTOR TEMPERATURA

factor_temp = np.where(X_train["Temperatura"] > TEMP_LIMITE,1.10,1.0)


# VARIABILIDAD OPERACIONAL

factor_operacional = np.random.uniform(0.85,1.15,N)

# TARGET FINAL

Y_train = (desgaste_base * factor_termico * factor_temp * factor_operacional)

print("Dataset sintético generado")

Dataset sintético generado


In [93]:
# MODELO RANDOM FOREST

modelo_rf = RandomForestRegressor(n_estimators=150,max_depth=8,random_state=42)
modelo_rf.fit(X_train, Y_train)
pred = modelo_rf.predict(X_train)

r2 = r2_score(Y_train, pred)
mae = mean_absolute_error(Y_train, pred)

print(" RESULTADOS MODELO PREDICTIVO")

print(f"R² Score: {r2:.4f}")

print(f"MAE:      {mae:.4f}")

 RESULTADOS MODELO PREDICTIVO
R² Score: 0.9823
MAE:      3.4194


## 4. Escenario Base: Asignación Fija

Se simulará un escenario operacional tradicional donde los camiones permanecen asignados permanentemente al mismo tajo.

Este escenario servirá como línea base de comparación para evaluar posteriormente el impacto de una estrategia inteligente de rotación dinámica.

In [94]:
# ESTADO ACTUAL

df_flota["Desgaste_Actual_mm"] = modelo_rf.predict(

    pd.DataFrame({"Horas_Uso": df_flota["Horas_Uso_Actual"],"Es_Tajo_Sur": np.where(df_flota["Asignacion_Actual"] == "Sur",1,0),
        "TKPH": df_flota["TKPH"],
        "Temperatura": df_flota["Temperatura"]
    })
)

df_flota.head()

,Camion_ID,Horas_Uso_Actual,Asignacion_Actual,TKPH,Temperatura,Desgaste_Actual_mm
0,CAT_001,2810.890416,Norte,626.199135,97.815819,17.527984
1,CAT_002,4827.500072,Norte,658.128879,92.051165,31.755932
2,CAT_003,4061.978796,Sur,804.315674,108.108385,32.736040
3,CAT_004,3595.304695,Norte,624.997700,94.310517,23.812910
4,CAT_005,2046.065242,Norte,715.302108,92.995298,13.952136


In [95]:
# ESCENARIO FIJO

df_flota["Desgaste_Final_Fijo_mm"] = modelo_rf.predict(

    pd.DataFrame({"Horas_Uso":df_flota["Horas_Uso_Actual"]+ HORAS_SIMULACION,"Es_Tajo_Sur": np.where(df_flota["Asignacion_Actual"] == "Sur",1,0),
        "TKPH": df_flota["TKPH"],
        "Temperatura": df_flota["Temperatura"]
    })
)

df_flota["mm_Consumidos_Fijo"] = (df_flota["Desgaste_Final_Fijo_mm"] - df_flota["Desgaste_Actual_mm"]
)
print("Escenario fijo calculado")

Escenario fijo calculado


## 5. Escenario Inteligente de Rotación Dinámica

Se implementará una lógica de asignación inteligente basada en:

- desgaste acumulado,
- TKPH,
- temperatura,
- y vida útil remanente.

La estrategia buscará reducir la severidad operacional de neumáticos críticos mediante rotación hacia zonas menos agresivas.

In [96]:
# REGLA IA

df_flota["Asignacion_Optimizada"] = np.where(((df_flota["Asignacion_Actual"] == "Sur") & (df_flota["Desgaste_Actual_mm"] > (LIMITE_DESGASTE_MM * 0.5)))
                                    |(df_flota["TKPH"] > TKPH_LIMITE)
                                    |( df_flota["Temperatura"] > TEMP_LIMITE),"Norte",df_flota["Asignacion_Actual"])


# NUEVO DESGASTE

df_flota["Desgaste_Final_Opt_mm"] = modelo_rf.predict(

                                                        pd.DataFrame({"Horas_Uso": df_flota["Horas_Uso_Actual"] + HORAS_SIMULACION,

                                                         "Es_Tajo_Sur": np.where(df_flota["Asignacion_Optimizada"] == "Sur", 1, 0),

                                                         "TKPH": np.where( df_flota["Asignacion_Optimizada"] == "Sur", 820, 650),

                                                         "Temperatura": np.where(df_flota["Asignacion_Optimizada"] == "Sur", 110, 92 )
                                                        })
)

df_flota["mm_Consumidos_Opt"] = (df_flota["Desgaste_Final_Opt_mm"] - df_flota["Desgaste_Actual_mm"])

print("Escenario IA calculado")

Escenario IA calculado


In [97]:
# COSTO POR MM

costo_por_mm = (COSTO_SET_LLANTAS / LIMITE_DESGASTE_MM)

# COSTOS

costo_fijo_total = (df_flota["mm_Consumidos_Fijo"].sum() * costo_por_mm)

costo_opt_total = (df_flota["mm_Consumidos_Opt"].sum() * costo_por_mm)


# AHORRO

ahorro = (costo_fijo_total- costo_opt_total)

porcentaje_ahorro = (ahorro / costo_fijo_total) * 100


# CAMIONES ROTADOS

camiones_rotados = len( df_flota[ df_flota["Asignacion_Actual"] != df_flota["Asignacion_Optimizada"]])


# RESULTADOS

print("="*60)
print(" RESULTADOS FINANCIEROS")
print("="*60)

print(f"OPEX Escenario Fijo: "f"${costo_fijo_total:,.2f} USD")
print( f"OPEX Escenario Inteligente: "f"${costo_opt_total:,.2f} USD")
print("-"*60)
print(f"Ahorro estimado: "f"${ahorro:,.2f} USD")
print(f"Reducción OPEX: "f"{porcentaje_ahorro:.2f}%")
print("-"*60)
print(f"Camiones rotados: "f"{camiones_rotados}")

 RESULTADOS FINANCIEROS
OPEX Escenario Fijo: $12,460,597.13 USD
OPEX Escenario Inteligente: $9,611,253.52 USD
------------------------------------------------------------
Ahorro estimado: $2,849,343.61 USD
Reducción OPEX: 22.87%
------------------------------------------------------------
Camiones rotados: 23


In [98]:
import plotly.graph_objects as go

# CAMIONES ROTADOS

df_rotados = df_flota[df_flota["Asignacion_Actual"]!=df_flota["Asignacion_Optimizada"]].copy()

# CÁLCULO DE AHORRO
df_rotados["Costo_Fijo_USD"] = (df_rotados["mm_Consumidos_Fijo"]* costo_por_mm)
df_rotados["Costo_Optimizado_USD"] = (df_rotados["mm_Consumidos_Opt"]* costo_por_mm)
df_rotados["Ahorro_USD"] = (df_rotados["Costo_Fijo_USD"]-df_rotados["Costo_Optimizado_USD"])

# TABLA FINAL

tabla = df_rotados[["Camion_ID","Asignacion_Actual","Asignacion_Optimizada","Ahorro_USD"

]].copy()

tabla["Ahorro_USD"] = (tabla["Ahorro_USD"].round(2))

# FIGURA TIPO POWER BI

fig = go.Figure(

    data=[
        go.Table(
            header=dict(
                values=[
                    "<b>Camión</b>",
                    "<b>Tajo Actual</b>",
                    "<b>Tajo Optimizado</b>",
                    "<b>Ahorro USD</b>"
                ],

                fill_color="#1F4E79",
                align="center",
                font=dict(color="white",size=14),
                height=35
            ),

            cells=dict(
                values=[
                    tabla["Camion_ID"],
                    tabla["Asignacion_Actual"],
                    tabla["Asignacion_Optimizada"],
                    tabla["Ahorro_USD"]
                ],
                fill_color=[["#F9F9F9", "#FFFFFF"] * 20],
                align="center",
                font=dict(size=13),
                height=30
            )
        )
    ]
)


# LAYOUT

fig.update_layout(
    title={
        "text":
        "Camiones Rotados por Optimización Inteligente",
        "x": 0.5,
        "font": {"size": 22}
    },
    height=700,
    width=950,
    margin=dict(l=20,r=20,t=60,b=20)
)

# MOSTRAR

fig.show()

### 5.1 KPIs Financieros del Modelo Inteligente de Optimización

A continuación, se presentan los indicadores ejecutivos del modelo híbrido de Inteligencia Artificial + Optimización Operacional.

La simulación compara:

- asignación fija tradicional,
- versus asignación dinámica inteligente.

Los KPIs permiten evaluar:

- reducción de OPEX,
- ahorro económico,
- y cantidad de camiones optimizados por el algoritmo.

In [99]:
import pandas as pd
import plotly.express as px

# DATAFRAME

df_costos = pd.DataFrame({"Escenario": [ "Asignación Fija", "Asignación Inteligente" ],"OPEX_USD": [ costo_fijo_total, costo_opt_total]})

# DIFERENCIA %

reduccion = ((costo_fijo_total-costo_opt_total)/costo_fijo_total) * 100

# GRÁFICO

fig = px.bar(

    df_costos,
    x="Escenario",
    y="OPEX_USD",
    text="OPEX_USD",
    template="plotly_white",
    title="Comparación de OPEX entre Estrategias Operacionales"
)

# PERSONALIZACIÓN

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    title_x=0.5,
    title_font_size=24,
    xaxis_title="Modelo Operacional",
    yaxis_title="Costo Operacional (USD)",
    height=650,
    width=1000,
    font=dict(size=15)
)

# ANOTACIÓN

fig.add_annotation(

    x="Asignación Inteligente",
    y=costo_opt_total,
    text=(f"<b>{reduccion:.2f}%</b><br>"f"reducción OPEX"),
    showarrow=True,
    arrowhead=2,
    yshift=40,
    bordercolor="black",
    borderwidth=1,
    bgcolor="white"
)

# MOSTRAR

fig.show()

In [100]:
import plotly.graph_objects as go

# FIGURA KPI EJECUTIVA
fig = go.Figure()

# KPI 1 — OPEX ESCENARIO FIJO

fig.add_trace(
    go.Indicator(
        mode="number",
        value=costo_fijo_total,
        title={ "text": "<b>OPEX Escenario Fijo</b><br>" "<span style='font-size:15px'>USD</span>"},
        number={ "prefix": "$", "valueformat": ",.0f"},
        domain={ "x": [0.00, 0.30], "y": [0.55, 1]}
    )
)

# KPI 2 — OPEX OPTIMIZADO

fig.add_trace(
    go.Indicator(
        mode="number",
        value=costo_opt_total,
        title={ "text": "<b>OPEX Inteligente</b><br>""<span style='font-size:15px'>USD</span>"},
        number={ "prefix": "$","valueformat": ",.0f"},
        domain={ "x": [0.35, 0.65], "y": [0.55, 1] }
    )
)

# KPI 3 — AHORRO

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=ahorro,
        delta={"reference": costo_fijo_total, "relative": True, "valueformat": ".1%"},
        title={ "text": "<b>Ahorro Estimado</b><br>" "<span style='font-size:15px'>USD</span>"},
        number={ "prefix": "$","valueformat": ",.0f"},
        domain={ "x": [0.70, 1.0], "y": [0.55, 1]}
    )
)

# KPI 4 — REDUCCIÓN OPEX

fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=porcentaje_ahorro,
        title={ "text": "<b>Reducción OPEX</b>"},
        number={ "suffix": "%" },
        gauge={ "axis": { "range": [0, 40]},"bar": { "thickness": 0.6}},
        domain={ "x": [0.05, 0.45], "y": [0.0, 0.40]}
    )
)

# KPI 5 — CAMIONES ROTADOS

fig.add_trace(
    go.Indicator(
        mode="number",
        value=camiones_rotados,
        title={ "text": "<b>Camiones Reasignados</b>"},
        number={"suffix": " equipos"},
        domain={ "x": [0.55, 0.95], "y": [0.0, 0.40]}
    )
)

# LAYOUT

fig.update_layout(
    title={
        "text":
        "Dashboard Ejecutivo — Optimización Inteligente de Neumáticos",
        "x": 0.5,
        "font": { "size": 24 }
    },
    template="simple_white",
    height=700,
    width=1200,
    font=dict(size=16),
    margin=dict( t=100, b=40, l=40, r=40
    )
)

# MOSTRAR

fig.show()

## Interpretación Técnica

Los resultados muestran que una estrategia de asignación dinámica inteligente permite:

- reducir la degradación acelerada de neumáticos,
- disminuir la variabilidad operacional,
- balancear la severidad de operación entre tajos,
- y reducir el costo operacional proyectado.

La integración de variables:
- térmicas,
- geomecánicas,
- y operacionales

demuestra la viabilidad técnica de implementar sistemas inteligentes de dispatch predictivo en operaciones de minería a tajo abierto.